# 🎙️ Channel 99 AI Studio — VoxCPM (Colab)
**Run all (Ctrl+F9)** → share URL ကို copy → recap tool ထဲ ကပ်.
Runtime: **T4 GPU** (Runtime → Change runtime type)
**v10**: `/generate_preset_timed` ထပ်ပါ — အသံ + word timestamps တခါတည်း�ြန် (tool က local whisper ကျော်နိုင်). အဟောင်း API မပြောင်း.


In [ ]:
# — ကိုယ့်အသံ ဒေါင်း (Drive link, login မလို) —
import urllib.request, os
VOICE_PATH = "/tmp/my_voice.wav"
if not os.path.exists(VOICE_PATH):
    print("⬇️ my_voice.wav ဒေါင်းနေပြီ...")
    urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1zTw0hH17OnZoN8I28SMNnkS02eyTTH8n&confirm=t", VOICE_PATH)
    print(f"✅ {os.path.getsize(VOICE_PATH)/1024:.0f} KB")
else:
    print("✅ my_voice.wav ready")

# — Alex preset အသံ ဒေါင်း (Drive link, login မလို) —
ALEX_PATH = "/tmp/alex_voice.wav"
if not os.path.exists(ALEX_PATH):
    print("⬇️ alex_voice.wav ဒေါင်းနေပြီ...")
    try:
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1r0dg-Iv9hCVYEwl0PkqoZcJWckB4Lewi&confirm=t", ALEX_PATH)
        print(f"✅ {os.path.getsize(ALEX_PATH)/1024:.0f} KB")
    except Exception as e:
        print(f"❌ alex_voice.wav ဒေါင်းမရပါ ({e}) — Drive sharing (Anyone with the link) စစ်ပါ")
else:
    print("✅ alex_voice.wav ready")

# — Sulafat preset အသံ ဒေါင်း (Drive link, login မလို) —
SULAFAT_PATH = "/tmp/sulafat_voice.wav"
if not os.path.exists(SULAFAT_PATH):
    print("⬇️ sulafat_voice.wav ဒေါင်းနေပြီ...")
    try:
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1VPaZ3lMqlOCm4-eHm95wMTdwvN1qRXHd&confirm=t", SULAFAT_PATH)
        print(f"✅ {os.path.getsize(SULAFAT_PATH)/1024:.0f} KB")
    except Exception as e:
        print(f"❌ sulafat_voice.wav ဒေါင်းမရပါ ({e}) — Drive sharing (Anyone with the link) စစ်ပါ")
else:
    print("✅ sulafat_voice.wav ready")

# — Sisi preset အသံ ဒေါင်း (Drive link, login မလို) —
SISI_PATH = "/tmp/sisi_voice.wav"
if not os.path.exists(SISI_PATH):
    print("⬇️ sisi_voice.wav ဒေါင်းနေပြီ...")
    try:
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1ogAhKGQ_9LBjjUNCqFP8QhmQmeV4Ntil&confirm=t", SISI_PATH)
        print(f"✅ {os.path.getsize(SISI_PATH)/1024:.0f} KB")
    except Exception as e:
        print(f"❌ sisi_voice.wav ဒေါင်းမရပါ ({e}) — Drive sharing (Anyone with the link) စစ်ပါ")
else:
    print("✅ sisi_voice.wav ready")


In [ ]:
# Install (ပထမအကြိမ်ပဲ ကြာ, နောက် run တွေ cache)
!pip install -q voxcpm soundfile gradio gradio_client faster-whisper


In [ ]:
import os
# — GPU စစ် (T4 မရရင် ရှင်းရှင်းပြ) —
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError(
        "❌ GPU မတွေ့!\n"
        "1) Runtime → Change runtime type → T4 GPU ရွေးပြီး ပြန် run\n"
        "2) 'GPU quota exhausted' ဆို: account ပြောင်း (သို့) Kaggle သုံး"
    )

# torch အသစ် (Colab Python 3.13) မှာ dynamo disable + voxcpm ရဲ့
# fullgraph compile တွဲမိရင် RuntimeError တက်လို့ compile ကို eager
# no-op ပြောင်းထား (2026-11-21 fix) — အမြန်နှုန်း မကွာသလောက်ပဲ
torch._dynamo.config.suppress_errors = True
def _eager_compile(fn=None, *args, **kwargs):
    return fn if fn is not None else (lambda g: g)
torch.compile = _eager_compile
from voxcpm import VoxCPM
import gradio as gr
import soundfile as sf
import numpy as np, gc
from datetime import datetime

# Model: Drive မှာ ရှိရင် Drive က, မရှိရင် HuggingFace ကနေ တိုက်ရိုက်ဒေါင်း
_drive_model = "/content/drive/MyDrive/Channel99_Studio/VoxCPM/models"
if os.path.isdir(_drive_model):
    _model_src = _drive_model
    print(f"📁 Drive model သုံးမယ်: {_drive_model}")
else:
    _model_src = "openbmb/VoxCPM2"
    print(f"☁️ Drive မှာ model မတွေ့ — HuggingFace ကနေ ဒေါင်းမယ်: {_model_src}")

print("⏳ Model ခေါ်နေပါပြီ...")
model = VoxCPM.from_pretrained(_model_src, load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"✅ Model ready! sample_rate={SR}")

# Output folder သေချာ ရှိနေအောင်
OUTPUT_DIR = "/tmp/voxcpm_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 outputs ready: {OUTPUT_DIR}")





In [ ]:
# ── ကိုယ့်အသံ preset ──
# my_voice.wav ကို Drive: /Channel99_Studio/ ထဲ တင်
# prompt_text = ဖိုင်ထဲမှာ ပြောထားတဲ့ စာသား အတိအကျ
VOICE_DATABASE = {
    "🎙️ ကိုယ့်အသံ": {
        "audio_path": "/tmp/my_voice.wav",
        "prompt_text": "ရန်ကုန်စည်ပင်နယ်နိမိတ်အတွင်း ဓာတ်လှေကားပါဝင်မယ့် ဆောက်လုပ်မယ့် အဆောက်အဦးတွေနဲ့ ဆောက်လုပ်ဆဲ အဆောက်အဦးတွေမှာ ဆိုလာဓာတ်အားပေးစနစ် မဖြစ်မနေ ထည့်သွင်းတပ်ဆင်ရမယ်လို့ စည်ပင်သာယာရေးကော်မတီ အဆောက်အအုံအင်ဂျင်နီယာဌာနက ညွှန်ကြားချက် ထုတ်ပြန်ပါတယ်။ ဆောက်လုပ်ရေးပစ္စည်းတွေ ဈေးကြီး၊ လုပ်ငန်းအများစု ရပ်နားထားရချိန်နဲ့ ဆိုလာနဲ့ ဆက်စပ်ပစ္စည်းတွေ ဈေးအဆမတန် တက်နေချိန်မှာ ဈေးကွက်ကို မထိန်းညှိနိုင်ဘဲ ဆိုလာစနစ် မဖြစ်မနေ ထည့်သွင်းတပ်ဆင်ခိုင်းတာက ဘယ်လောက်ထိ အခက်အခဲ ဖြစ်စေနိုင်ပါသလဲ။ မဖြစ်မနေ တပ်ဆင်ရမယ်ဆိုရင် ဘာတွေလုပ်ဖို့ လိုအပ်ပါသလဲ။ ဆိုလာနည်းပညာ ကျွမ်းကျင်သူတွေနဲ့ ဆောက်လုပ်ရေး လုပ်ငန်းရှင်တွေက ဒီအပေါ်မှာ ဘယ်လို ရှုမြင်ကြပါသလဲ။ အပြည့်အစုံကို မသူဇာလင်းက ပြောပြမှာပါ။",
    },
    "🎙️ Alex": {
        "audio_path": "/tmp/alex_voice.wav",
        "prompt_text": "Make heads or tails of this. You say this when you can't understand something because it's confusing. You say, I've read this three times and I still can't make heads or tails of it. I'm as much in the dark as you are. You say this when you know just as little about a situation as the other person. You say, why was the meeting cancelled? I'm as much in the dark as you are. I think it's worth a shot. You say this when you think something is worth trying even if it might not work. You say, we might get rejected, but I think it's worth a shot. If I play my cards right, you say this when you believe you can get what you want by making the right choices. You say, if I play my cards right I could get that promotion. We hit it off right away. You say this when two people like each other and get along from the beginning. You say, I was nervous about meeting her, but we hit it off right away. Take a rain check on that. You say this when you want to postpone an invitation or activity until another time. You say, I'd love to, but can I take a rain check on that? I'm exhausted.",
    },
    "🎙️ Sulafat": {
        "audio_path": "/tmp/sulafat_voice.wav",
        "prompt_text": "Three thousand years ago, in the land of Egypt, a man stood on the deck of a ship and opened a small wooden box. Inside was a bird. He lifted it gently into the sky, and the bird flew straight and fast toward the land. Tied to its leg was a tiny message: We are coming home. How did the bird know the way? Nobody could answer that question. Not the sailors of Egypt. Not the soldiers of Rome. And not even the scientists of today can explain all of it. But this small, ordinary bird carried the letters of kings, the secrets of armies, and the last hope of dying men. This is the story of the pigeon, the first messenger of the world.",
    },
    "🎙️ Sisi": {
        "audio_path": "/tmp/sisi_voice.wav",
        "prompt_text": "Break the ice When do we use it? When we help people feel comfortable and start talking at a social gathering. Example: \"He told a funny story to break the ice.\" Keep your head up When do we use it? When we encourage someone to stay confident and positive during a hard time. Example: \"Don't lose hope. Keep your head up!\" Take it easy When do we use it? When we tell someone to relax and not worry so much about something. Example: \"Why are you so stressed? Take it easy!\"",
    },
}


In [ ]:
# ── Prompt cache: reference voice ကို တစ်ခါတည်း encode, အပိုင်းတိုင်း ပြန်သုံး ──
# (Voice-Studio backend pattern — thelapyae/Voice-Studio)
_PROMPT_CACHE = {}

def get_prompt_cache(audio_path, prompt_text):
    key = (str(audio_path), str(prompt_text or ""))
    pc = _PROMPT_CACHE.get(key)
    if pc is None:
        pc = model.tts_model.build_prompt_cache(
            prompt_text=prompt_text,
            prompt_wav_path=audio_path,
            reference_wav_path=audio_path,
        )
        _PROMPT_CACHE[key] = pc
    return pc

def _to_numpy(wav):
    if hasattr(wav, "detach"):
        wav = wav.detach().cpu()
        if hasattr(wav, "squeeze"):
            wav = wav.squeeze(0)
        wav = wav.numpy()
    return np.asarray(wav, dtype=np.float32)

def run_tts(text_input, audio_path, prompt_text):
    gc.collect(); torch.cuda.empty_cache()
    smart = (text_input.replace('။', '။\n').replace('.', '.\n')
                       .replace('?', '?\n').replace('!', '!\n'))
    chunks = [t.strip() for t in smart.split('\n') if t.strip()]
    if not chunks:
        return None
    # Prompt cache ရမရ စမ်း (VoxCPM version အလိုက် API ကွဲနိုင် → မရရင် ပုံမှန်လမ်း)
    try:
        _pc = get_prompt_cache(audio_path, prompt_text)
        _use_cache = hasattr(model.tts_model, "generate_with_prompt_cache")
        if _use_cache:
            print("⚡ prompt cache သုံးမယ် (reference တစ်ခါတည်း encode)")
    except Exception as e:
        print(f"⚠️ prompt cache မရ ({e}) — ပုံမှန် generate သုံးမယ်")
        _use_cache = False
    silence = np.zeros(int(SR * 0.15), dtype=np.float32)
    wavs = []
    for i, ch in enumerate(chunks):
        if len(ch) < 2:
            continue
        # seed ချုပ် → run တိုင်း အသံတူ (model.generate က seed kwarg မရ)
        torch.manual_seed(42); torch.cuda.manual_seed_all(42); np.random.seed(42)
        with torch.inference_mode():
            if _use_cache:
                wt, _, _ = model.tts_model.generate_with_prompt_cache(
                    target_text=ch + " ",
                    prompt_cache=_pc,
                    inference_timesteps=15,
                    cfg_value=2.1,
                    max_len=2048,
                )
                w = _to_numpy(wt)
            else:
                w = model.generate(text=ch + " ", prompt_wav_path=audio_path,
                                   prompt_text=prompt_text,
                                   cfg_value=2.1, inference_timesteps=15)
        wavs.append(w)
        if i < len(chunks) - 1:
            wavs.append(silence)
        torch.cuda.empty_cache(); gc.collect()
    return np.concatenate(wavs)


In [ ]:
import re as _re

def _fname_from_text(text, ts):
    """Output filename: first ~6 words of the input text + timestamp."""
    words = _re.findall(r"[A-Za-z0-9']+", str(text or ""))[:6]
    base = "_".join(words) if words else "voice"
    return f"{base}_{ts}.wav"

def generate_preset(text_input, voice_choice):
    os.makedirs("/tmp/voxcpm_outputs", exist_ok=True)
    sel = VOICE_DATABASE.get(voice_choice, list(VOICE_DATABASE.values())[0])
    wav = run_tts(text_input, sel["audio_path"], sel["prompt_text"])
    if wav is None:
        return None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = f"/tmp/voxcpm_outputs/{_fname_from_text(text_input, ts)}"
    sf.write(out, wav, SR)
    return out

def generate_clone(text_input, uploaded_audio, reference_text):
    if uploaded_audio is None:
        return None, "⚠️ အသံဖိုင် တင်ပေးပါ။"
    if not (reference_text or "").strip():
        return None, "⚠️ Reference Text ထည့်ပေးပါ။"
    wav = run_tts(text_input, uploaded_audio, reference_text.strip())
    if wav is None:
        return None, "⚠️ စာသား မပါပါ။"
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = f"/tmp/voxcpm_outputs/{_fname_from_text(text_input, ts)}"
    sf.write(out, wav, SR)
    return out, "✅ အသံထုတ်ပြီးပါပြီ!"


# ── v10: timed generation (အသံ + word timestamps တခါတည်း) ──
# Recap tool's local VoxCPMEngine._word_boundaries နဲ့ ပုံစံအတူတူ:
# faster-whisper (GPU), language="my", Edge-style [{text, offset, duration}]
import json as _json

_WHISPER = None

def _get_whisper():
    global _WHISPER
    if _WHISPER is None:
        # CTranslate2 must find the bundled CUDA libs (libcublas/libcudnn
        # live in the nvidia-* wheel dirs, off the loader path on some
        # images -> "Library libcublas.so.12 is not found", Kaggle
        # live run 2026-11-06). Prepend them to LD_LIBRARY_PATH first.
        import glob as _glob, os as _os, site as _site
        _roots = []
        try: _roots += _site.getsitepackages()
        except Exception: pass
        _found = []
        for _r in _roots:
            _found += [d for d in _glob.glob(_os.path.join(_r, "nvidia", "*", "lib")) if _os.path.isdir(d)]
            _tl = _os.path.join(_r, "torch", "lib")
            if _os.path.isdir(_tl): _found.append(_tl)
        if _found:
            _parts = [p for p in _os.environ.get("LD_LIBRARY_PATH", "").split(":") if p]
            for _d in _found:
                if _d not in _parts: _parts.insert(0, _d)
            _os.environ["LD_LIBRARY_PATH"] = ":".join(_parts)
        # LD_LIBRARY_PATH alone is not enough (loader path is fixed at
        # process start) -> preload cuBLAS/cuDNN by absolute path so
        # CTranslate2's soname lookup hits the loaded libs (Kaggle run
        # #3, 2026-11-06).
        import ctypes as _ct
        for _name in ("libcublasLt.so.12", "libcublas.so.12", "libcudnn.so.9", "libcudnn.so.8"):
            for _d in _found:
                _p = _os.path.join(_d, _name)
                if _os.path.exists(_p):
                    try: _ct.CDLL(_p, mode=_ct.RTLD_GLOBAL)
                    except OSError: pass
                    break
        from faster_whisper import WhisperModel
        print("⏳ Whisper (small) loading for word timings...")
        _WHISPER = WhisperModel("small", device="cuda", compute_type="float16")
    return _WHISPER

def _to_16k(a, sr):
    import numpy as _np
    if sr == 16000 or a.size == 0:
        return a
    n = int(round(a.size * 16000 / sr))
    x = _np.linspace(0.0, a.size - 1, num=n)
    return _np.interp(x, _np.arange(a.size), a).astype(_np.float32)

def _load_wav_16k_mono(wav_path):
    # PyAV-free WAV decode: some Kaggle/Colab PyAV builds reject the
    # metadata_errors kwarg faster-whisper passes to av.open (TypeError,
    # first seen on Kaggle live run 2026-11-06). Feeding transcribe() a
    # float32 mono 16kHz array skips PyAV entirely.
    import numpy as _np
    import wave as _wave
    try:
        with _wave.open(str(wav_path), "rb") as w:
            sr = w.getframerate(); ch = w.getnchannels(); sw = w.getsampwidth()
            raw = w.readframes(w.getnframes())
    except Exception:
        import soundfile as _sf
        data, sr = _sf.read(str(wav_path), dtype="float32", always_2d=True)
        return _to_16k(data.mean(axis=1).astype(_np.float32), sr)
    if sw == 1:
        a = (_np.frombuffer(raw, dtype=_np.uint8).astype(_np.float32) - 128.0) / 128.0
    elif sw == 2:
        a = _np.frombuffer(raw, dtype=_np.int16).astype(_np.float32) / 32768.0
    elif sw == 3:
        b = _np.frombuffer(raw, dtype=_np.uint8).reshape(-1, 3)
        v = (b[:, 0].astype(_np.int32) | (b[:, 1].astype(_np.int32) << 8) | (b[:, 2].astype(_np.int32) << 16))
        v = _np.where(v >= 1 << 23, v - (1 << 24), v)
        a = v.astype(_np.float32) / 8388608.0
    elif sw == 4:
        a = _np.frombuffer(raw, dtype=_np.int32).astype(_np.float32) / 2147483648.0
    else:
        raise ValueError(f"unsupported WAV sample width: {sw}")
    if ch > 1:
        a = a.reshape(-1, ch).mean(axis=1)
    return _to_16k(a.astype(_np.float32), sr)

def _word_boundaries(wav_path):
    model = _get_whisper()
    bounds = []
    segments, _info = model.transcribe(_load_wav_16k_mono(wav_path), word_timestamps=True, language="my")
    for seg in segments:
        for w in (seg.words or []):
            bounds.append({"text": str(w.word or "").strip(),
                           "offset": float(w.start or 0.0),
                           "duration": max(0.0, float(w.end or 0.0) - float(w.start or 0.0))})
    return [b for b in bounds if b["text"]]

def generate_preset_timed(text_input, voice_choice):
    out = generate_preset(text_input, voice_choice)
    if out is None:
        return None, "[]"
    return out, _json.dumps(_word_boundaries(out), ensure_ascii=False)


In [ ]:
with gr.Blocks(title="🎙️ Channel 99 AI Studio") as demo:
    gr.Markdown("# 🎙️ Channel 99 AI Studio")
    with gr.Tabs():
        with gr.Tab("🎭 Preset Voice"):
            with gr.Row():
                with gr.Column():
                    t1 = gr.Textbox(lines=10, label="📝 စာသားထည့်ပါ")
                    v1 = gr.Radio(choices=list(VOICE_DATABASE.keys()),
                                  value=list(VOICE_DATABASE.keys())[0],
                                  label="🗣️ အသံရွေးပါ")
                    b1 = gr.Button("🎙️ အသံထုတ်မယ်", variant="primary")
                with gr.Column():
                    o1 = gr.Audio(label="🎧 ထွက်လာမယ့် အသံ")
            b1.click(fn=generate_preset, inputs=[t1, v1], outputs=o1)
            with gr.Accordion("🔌 Tool API — timed (WAV + word timestamps)", open=False):
                bt = gr.Button("🎙️⏱️ Timed generate (API စမ်းရန်)")
                ot = gr.Audio(label="🎧 အသံ")
                jt = gr.Textbox(label="⏱️ Word boundaries (JSON)", lines=4)
                bt.click(fn=generate_preset_timed, inputs=[t1, v1], outputs=[ot, jt], api_name="generate_preset_timed")
        with gr.Tab("🎤 Clone Voice"):
            with gr.Row():
                with gr.Column():
                    t2 = gr.Textbox(lines=10, label="📝 စာသားထည့်ပါ")
                    a2 = gr.Audio(label="🎵 Reference အသံဖိုင် (WAV)", type="filepath")
                    r2 = gr.Textbox(lines=4, label="📄 Reference Text (အတိအကျ)")
                    b2 = gr.Button("🎤 Clone ပြီး ထုတ်မယ်", variant="primary")
                with gr.Column():
                    o2 = gr.Audio(label="🎧 ထွက်လာမယ့် အသံ")
                    s2 = gr.Textbox(label="Status", interactive=False)
            b2.click(fn=generate_clone, inputs=[t2, a2, r2], outputs=[o2, s2])

import threading, time

print("\n🚀 Channel 99 AI Studio — 2 tabs: Preset Voice | Clone Voice")
print("📌 Colab tab ကို ဖွင့်ထားပါ (ပိတ်လိုက်ရင် server ရပ်မယ်)\n")


def _launch():
    demo.launch(share=True, debug=False, prevent_thread_lock=True)

t = threading.Thread(target=_launch, daemon=True)
t.start()

share_url = None
for _ in range(60):  # max ~5 min စောင့်
    time.sleep(5)
    if getattr(demo, "share_url", None):
        share_url = demo.share_url.strip()
        break

if share_url:
    print("\n" + "="*60 + "\n📋 COPY THIS URL → recap tool:\n" + share_url + "\n" + "="*60)
else:
    print("⚠️ share URL မရသေး — output အပေါ်က 'Running on public URL' ကို ကြည့်")

t.join()  # server ဆက်ရှင်နေအောင်


In [ ]:
# ── Share URL ကို ဒီမှာ ကြည့်ပါ ──
# အပေါ် cell ရဲ့ output မှာ "Running on public URL: https://xxxxx.gradio.live" ပါတယ်
# အဲ့ URL ကို copy → recap tool Config → VoxCPM URL ထဲ ကပ်
